Este bloco faz a requisição para a API da Open Library e obtém uma lista das 13 categorias de livros em formato json

In [9]:
import sqlite3
import requests

#URLS das 13 categorias
urls = [
"https://openlibrary.org/subjects/arts.json",
"https://openlibrary.org/subjects/animals.json",
"https://openlibrary.org/subjects/fiction.json",
"https://openlibrary.org/subjects/sciencemathematics.json",
"https://openlibrary.org/subjects/business.json",
"https://openlibrary.org/subjects/juvenile_fiction.json",
"https://openlibrary.org/subjects/history.json",
"https://openlibrary.org/subjects/health.json",
"https://openlibrary.org/subjects/biography.json",
"https://openlibrary.org/subjects/social.json",
"https://openlibrary.org/subjects/places.json",
"https://openlibrary.org/subjects/textbooks.json",
"https://openlibrary.org/subjects/language.json"
]

Esse trecho cria o banco de dados e as tabelas

In [10]:
#Conecta ao banco de dados SQLite
conn = sqlite3.connect("biblioteca.db")
cursor = conn.cursor()

#Cria a tabela categorias
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categorias (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE,
        work_count INTEGER
    )
""")

#Cria a tabela autores
cursor.execute("""
    CREATE TABLE IF NOT EXISTS autores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE
    )
""")

#Cria a tabela livros com coluna para os assuntos e autores
cursor.execute("""
    CREATE TABLE IF NOT EXISTS livros (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        categoria_id INTEGER,
        title TEXT,
        authors TEXT,  -- Apenas para visualização rápida (nomes separados por vírgulas)
        subjects TEXT,  -- Armazena os assuntos como string separada por vírgulas
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    )
""")

#Cria a tabela tabela de relacionamento entre livros e autores
cursor.execute("""
    CREATE TABLE IF NOT EXISTS livro_autor (
        livro_id INTEGER,
        autor_id INTEGER,
        PRIMARY KEY (livro_id, autor_id),
        FOREIGN KEY (livro_id) REFERENCES livros(id),
        FOREIGN KEY (autor_id) REFERENCES autores(id)
    )
""")

conn.commit()

Esse trecho extrai os dados das URLs e prepara eles para inserir no banco de dados

In [11]:
#para cada url em urls vai fazer uma requisição dos dados pra converter em uma lista 
for url in urls:
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro ao acessar: {url}")
        continue

    try:
        dados = response.json()
    except Exception as e:
        print(f"Erro ao converter JSON da URL {url}: {e}")
        continue

    name = dados.get("name", "Categoria Desconhecida")
    work_count = dados.get("work_count", 0)
    works = dados.get("works", [])

    #Verificação para garantir que a variável works seja uma lista
    if not isinstance(works, list):
        print(f"Formato inesperado para {url}, ignorando...")
        continue

    print(f"Processando categoria: {name} - {len(works)} livros encontrados")

    # Insere categoria na base de dados e evita a duplicação
    cursor.execute("SELECT id FROM categorias WHERE name = ?", (name,))
    categoria = cursor.fetchone()

    if categoria:
        categoria_id = categoria[0]
    else:
        cursor.execute("INSERT INTO categorias (name, work_count) VALUES (?, ?)", (name, work_count))
        categoria_id = cursor.lastrowid

    #Insere livros, autores e assuntos
    for livro in works:
        title = livro.get("title", "Sem título").strip()
        if not title:
            print("Livro sem título encontrado, ignorando...")
            continue

        #Extrai e cadastra os autores
        
        authors_list = []
        author_ids = []
        
        #Itera sobre a lista de autores do livro se houver.
        for author in livro.get("authors", []):
            #Obtém o nome do autor ou define como "Autor desconhecido" se não houver nome
            author_name = author.get("name", "Autor desconhecido").strip()

            # Verificar se o autor já existe
            cursor.execute("SELECT id FROM autores WHERE name = ?", (author_name,))
            autor = cursor.fetchone()

            if autor:
                autor_id = autor[0]
            else:
                cursor.execute("INSERT INTO autores (name) VALUES (?)", (author_name,))
                autor_id = cursor.lastrowid

            author_ids.append(autor_id)
            authors_list.append(author_name)

        authors = ", ".join(authors_list) if authors_list else "Autor desconhecido"

        # Extrair e formatar assuntos
        subjects_list = livro.get("subject", [])
        subjects = ", ".join(subjects_list) if subjects_list else "Sem assunto"

        # Inserir livro na tabela livros
        cursor.execute("""
            INSERT INTO livros (categoria_id, title, authors, subjects)
            VALUES (?, ?, ?, ?)
        """, (categoria_id, title, authors, subjects))
        livro_id = cursor.lastrowid

        # Relacionar livro com os autores
        for autor_id in author_ids:
            cursor.execute("""
                INSERT OR IGNORE INTO livro_autor (livro_id, autor_id)
                VALUES (?, ?)
            """, (livro_id, autor_id))

    conn.commit()

conn.close()
print("Todas as categorias, livros e autores foram inseridos com sucesso!")

Processando categoria: arts - 12 livros encontrados
Processando categoria: animals - 12 livros encontrados
Processando categoria: fiction - 12 livros encontrados
Processando categoria: sciencemathematics - 12 livros encontrados
Processando categoria: business - 12 livros encontrados
Processando categoria: juvenile fiction - 12 livros encontrados
Processando categoria: history - 12 livros encontrados
Processando categoria: health - 12 livros encontrados
Processando categoria: biography - 12 livros encontrados
Processando categoria: social - 12 livros encontrados
Processando categoria: places - 12 livros encontrados
Processando categoria: textbooks - 12 livros encontrados
Processando categoria: language - 12 livros encontrados
Todas as categorias, livros e autores foram inseridos com sucesso!
